# 🚀 EKVA v2: Routing-as-a-Signal for Sparse MoE KV Cache Compression
### Complete Google Colab Runner (T4 / A100 GPU)

This notebook runs the **EKVA v2** experimental evaluation:
1. **Environment Setup & GPU Verification**
2. **MoE Routing Signature Extraction Hooks** (`Mixtral-8x7B`, `Qwen1.5-MoE-A2.7B`, `DeepSeek-MoE-16B`)
3. **Multi-Signal Token Saliency Engine** ($S(x_t) = w_a \hat{A} + w_r R + w_s \text{Sink} + w_c \text{Recency}$)
4. **Cross-Signal Correlation Check** $\rho(R(x_t), \hat{A}(x_t))$
5. **Multi-Benchmark Evaluation** (GSM8K, HumanEval, PG19 PPL, NIAH Retrieval) vs. Baselines (FullKV, Uniform, H2O, SnapKV, CAKE)
6. **Real Live Model Inference** on Pretrained `Qwen1.5-MoE-A2.7B` on GSM8K prompts
7. **Fused Triton Compaction Kernel & Latency Profiling** (TTFT, TPOT, Speedup)
8. **Publication Plots Generation & Results Export**

In [ ]:
# Step 1: Install Dependencies
import subprocess, sys
packages = ["torch", "transformers", "datasets", "accelerate", "triton", "matplotlib", "seaborn", "tqdm", "pytest", "bitsandbytes"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)
subprocess.run(["nvidia-smi"])

In [ ]:
# Step 2: Setup Repository & Check Hardware
import os, torch
if not os.path.exists('EKVA') and not os.path.exists('ekva'):
    os.system('git clone https://github.com/GauravPatil2515/EKVA.git')
    os.chdir('EKVA')
elif os.path.exists('EKVA'):
    os.chdir('EKVA')

print(f"Current Directory: {os.getcwd()}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 3: Run Full Unit Test Suite
import pytest
pytest.main(["tests/", "-v"])

In [ ]:
# Step 4A: Execute Full EKVA v2 Multi-Benchmark Suite (All 3 Models)
from scripts.run_ekva_v2_experiments import run_full_evaluation_pipeline
results = run_full_evaluation_pipeline(out_dir="output")

In [ ]:
# Step 4B: (OPTIONAL) Run Real Live Pretrained Model Weights Inference on GSM8K
# Downloads real Qwen/Qwen1.5-MoE-A2.7B weights and executes real token eviction
from scripts.evaluate_real_hf_model import run_real_evaluation
run_real_evaluation(model_name="qwen1.5-moe-a2.7b", num_samples=20, out_dir="output")

In [ ]:
# Step 5: Display Publication Figures
import os
try:
    from IPython.display import Image, display
    if os.path.exists('output/fig2_ablation_curves.png'):
        print("\n📈 Figure 2: Retained Performance Curves across Budgets")
        display(Image('output/fig2_ablation_curves.png'))
    if os.path.exists('output/analytical_roofline.png'):
        print("\n📈 Figure 5: Analytical Roofline & Speedup Profile")
        display(Image('output/analytical_roofline.png'))
except ImportError:
    print("Figures generated in output/ directory.")

In [ ]:
# Step 6: Print Evaluation Summary Table
import json
if os.path.exists('output/ekva_v2_results.json'):
    with open('output/ekva_v2_results.json') as f:
        summary = json.load(f)
    print("=" * 65)
    print("EKVA v2 MULTI-BENCHMARK EVALUATION SUMMARY (40% Budget)")
    print("=" * 65)
    for model, mdata in summary.items():
        print(f"\nModel: {model.upper()} (rho = {mdata['correlation_rho']})")
        for task, tdata in mdata['tasks'].items():
            b40 = tdata['40%']
            print(f"  - {task:10s} | EKVA v2: {b40['A+R (EKVA v2)']['mean']} | SnapKV: {b40['SnapKV']['mean']} | H2O: {b40['H2O']['mean']} | CAKE: {b40['CAKE']['mean']}")

In [ ]:
# Step 7: Zip Output Artifacts for Download
import shutil, os
if os.path.exists('output'):
    shutil.make_archive('ekva_v2_colab_artifacts', 'zip', 'output')
    print("Created ekva_v2_colab_artifacts.zip")
    try:
        from google.colab import files
        files.download('ekva_v2_colab_artifacts.zip')
    except Exception:
        print("Download ekva_v2_colab_artifacts.zip from the Colab file browser.")